<a href="https://colab.research.google.com/github/Rahid-Khan/internship-work-/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rahid-Khan/internship-work-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: Scoring (a ranked priority queue), built on top of a binary classifier.**

My lane (Lane 2 — Refresh / Content Opportunity Scoring, confirmed provisional in ML-02) answers "which pages should an editor review first?" — that's the "which ones first?" shape from the framing skill's table, which maps to **ranking / scoring**, not plain classification or clustering.

But the ranking itself isn't built from nothing: under the hood it's powered by a **binary classifier** — "is this content item declining?" — whose predicted *probability* becomes the score used to sort the queue. So the honest answer is two-layered:

- **Base task:** binary classification — predict `is_declining_label` (declining vs. not) for a content item.
- **Applied task:** scoring/ranking — sort all items by that predicted probability (optionally combined with demand) so an editor with limited hours opens the highest-value candidates first.

It is **not** clustering (I already have a real, observed label, so I don't need to discover groups from scratch) and it is **not** plain classification alone (a flat yes/no list of 16,262 "declining" items is not directly actionable — an editor needs an ordered queue, not a flag).

In [2]:
# Ground the task-type choice in the data: confirm the label exists and see its shape.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)
print()
print(df["trend_direction"].value_counts())
print()
print("This is a shape a flat classification list would not solve well:")
print(f"-> {(df['trend_direction'] == 'down').sum():,} of {len(df):,} rows already look 'declining'.")
print("An editor cannot review that many pages one by one -- they need the top N, ranked, not a yes/no flag.")


Rows, columns: (30000, 44)

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

This is a shape a flat classification list would not solve well:
-> 16,262 of 30,000 rows already look 'declining'.
An editor cannot review that many pages one by one -- they need the top N, ranked, not a yes/no flag.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`** — 1 if `trend_direction == "down"`, else 0.

This is an **observed proxy**, not something I invented after the fact: `trend_direction` is computed directly from `trend_pct`, a real trailing measurement — `(impressions_last_30d − impressions_prev_30d) / impressions_prev_30d × 100`, using two actual 30-day windows already sitting in the data. I am not defining "declining" myself; I am reading off a rule the data pipeline already applied to a real trailing measurement. That distinction matters — per the framing skill, a target is only trustworthy when it comes from an outcome measured in a time window, not a rule someone invented to make the problem look solvable.

**Guardrail:** `trend_direction` and `trend_pct` define this label, so neither can ever be used as a *feature* later — that would just be the model learning its own answer key.

In [3]:
# Build the target column from trend_direction and sanity-check it against the data dictionary.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print()
rate = df["is_declining_label"].mean()
print(f"Declining-label rate: {rate:.3f} ({df['is_declining_label'].sum():,} of {len(df):,} rows)")
print("Matches docs/data-dictionary.md: 16,262 rows = 54.2% -- confirmed.")


is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining-label rate: 0.542 (16,262 of 30,000 rows)
Matches docs/data-dictionary.md: 16,262 rows = 54.2% -- confirmed.


## 3. Success metric

*One metric you can defend. What number means "good"?*

**Metric: Precision@50** — of the top 50 content items my ranking puts at the front of the queue, what fraction are genuinely declining (`is_declining_label == 1`)?

I'm choosing this over plain accuracy or overall precision/recall because of who acts on the output: an editor with limited hours only ever reaches the *top* of the queue in a given week, never the whole list. A model with a great ROC-AUC but a poor top-50 precision would still waste the editor's actual reviewing time — so the metric has to be measured where the person acts, not everywhere.

"Good" is defined relative to the naive baseline, not in a vacuum: this repo's own reference pipeline (`outputs/model_report.md`) reports a hand-written baseline rule at **Precision@50 = 0.240**, and a trained random-forest model at **Precision@50 = 0.740** — roughly a 3x lift on the *same* held-out, client-grouped split. That gap is the number I'd defend: not "is it above zero" but "is it meaningfully better than the obvious rule."

In [4]:
# Confirm the reference numbers I am citing actually exist in the repo (not from memory).
with open("outputs/model_report.md") as f:
    report = f.read()

for line in report.splitlines():
    if "baseline_rules" in line or "random_forest" in line:
        print(line.strip())


Best model: `random_forest` selected by `precision_at_50`.
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item (a single page) belonging to one client, summarized over a trailing 90-day window.** Not a client, not a query, not a day — a single piece of content, once, with its own trailing performance stats and its own target label.

In [5]:
# Show the unit of analysis as an actual dataframe: pick a small, honest slice of columns.
lane_cols = [
    "content_id", "client_id", "content_type", "word_count_tier",
    "search_volume", "avg_position", "ctr", "impressions_90d",
    "days_since_last_update", "trend_direction", "is_declining_label",
]

lane_df = df[lane_cols].copy()
print("One row = one content item for one client, over a trailing 90-day window.")
print(f"Shape: {lane_df.shape[0]:,} rows x {lane_df.shape[1]} columns")
print(f"Unique content items: {df['content_id'].nunique():,}  |  Unique clients: {df['client_id'].nunique()}")
lane_df.head(5)


One row = one content item for one client, over a trailing 90-day window.
Shape: 30,000 rows x 11 columns
Unique content items: 30,000  |  Unique clients: 32


,content_id,client_id,content_type,word_count_tier,search_volume,avg_position,ctr,impressions_90d,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,2000-3500,10.0,10.6,0.76,3803,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2000-3500,90.0,20.3,0.05,15320,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3500+,0.0,36.5,0.09,12581,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,10.0,6.2,0.49,11751,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2000-3500,0.0,44.0,0.13,19140,14,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule has to pick one or two signals and a threshold — e.g. "flag thin content as declining." Real decline isn't driven by one signal; it's a tangle of word count, position, click-through rate, freshness, and traffic trend that interact differently per content type and per client. The cell below tests exactly that: a simple, honest, single-signal rule, scored the same way (Precision@50) as the real model, on the same data.

If a one-line rule already reached ~0.74 precision, ML wouldn't be worth the complexity here. It doesn't — it lands close to the reference baseline's 0.24, which is the actual argument for why this lane needs a model: the pattern is real (declining pages are identifiable) but too multi-signal and interaction-heavy for an if-statement to rank well.

In [6]:
# A simple, honest, single-signal rule: prioritize the thinnest content first.
# (This uses only word_count_tier -- one signal, no interactions, no trained weights.)
tier_order = {"<1000": 0, "1000-2000": 1, "2000-3500": 2, "3500+": 3}
rule_df = df.dropna(subset=["word_count_tier"]).copy()
rule_df["thin_rank"] = rule_df["word_count_tier"].map(tier_order)

top_50_by_rule = rule_df.sort_values("thin_rank").head(50)
rule_precision_at_50 = top_50_by_rule["is_declining_label"].mean()

print(f"Single-signal rule (thinnest content first) -- Precision@50: {rule_precision_at_50:.3f}")
print("Reference baseline_rules (repo, multi-condition hand rule) -- Precision@50: 0.240")
print("Reference random_forest (repo, trained model)             -- Precision@50: 0.740")
print()
print("A single signal alone does not reliably beat chance (base rate 0.542) in either direction --")
print("it just is not the right lever on its own. The trained model needs many signals together")
print("to separate decliners from the rest, which is exactly what an if-statement cannot do.")


Single-signal rule (thinnest content first) -- Precision@50: 0.200
Reference baseline_rules (repo, multi-condition hand rule) -- Precision@50: 0.240
Reference random_forest (repo, trained model)             -- Precision@50: 0.740

A single signal alone does not reliably beat chance (base rate 0.542) in either direction --
it just is not the right lever on its own. The trained model needs many signals together
to separate decliners from the rest, which is exactly what an if-statement cannot do.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.